In [1]:
pip install pyserial

Note: you may need to restart the kernel to use updated packages.


In [2]:
import serial
print(serial.Serial)



<class 'serial.serialposix.Serial'>


In [8]:
%matplotlib inline
import pandas as pd
import pp

In [1]:
import sys
sys.path.insert(0, "../../TOMBAK/base-folder/build/lib/aerodiode/")
from tombak import Tombak
from aerodiode import Aerodiode


class pp():
    def __init__(self):
        """ init funtion to initialize tombak in divider mode
        
        Keyword arguments:
        none -- none so far
        """
        # fixed serial port
        self.tombak = Tombak('/dev/serial/by-id/usb-FTDI_TTL-232R-3V3-AJ_FTCAVMZN-if00-port0')
        # set tombak to divder mode needs to be set
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_FUNCTIONING_MODE, self.tombak.DIVIDER)
        # set tombak to fixed pulse width 5 ns
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_OUT_WIDTH, 5)
        # set to direct (not dsaisy chain) mode
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_PULSE_IN_SRC, self.tombak.DIRECT)
        # set trigger to internal since we dont use it
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_TRIGGER_SRC, self.tombak.INT)
        # set monitor out on to see pulse in osci
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_SYNC_OUT_1_SRC, self.tombak.PULSE_OUT)#ok
        # initialize divder
        self.divider = 10
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_IN_FREQUENCY_DIV, self.divider)
        #don't forget to apply settings
        self.tombak.apply_all()
        
    def get_inputFreqency(self):
        """ get input frequency (eg laser)
        
        Keyword arguments:
        returns -- frequency in Hz
        """
        return self.tombak.measure_pulse_in_frequency()
    
    def get_outputFreqency(self):
        """ get output frequency
        
        Keyword arguments:
        returns -- frequency in Hz
        """
        return self.tombak.measure_pulse_in_frequency()/self.divider
    
    def set_outputDivider(self, div):
        """ set output frequency divider
        
        Keyword arguments:
        div -- divider
        returns -- divided frequency in Hz
        """
        self.divider = div
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_IN_FREQUENCY_DIV, div)
        self.tombak.apply_all()
        return self.get_outputFreqency()
    
        
        """ set output pulse width in ns
        
        Keyword arguments:
        w -- pulse length in ns
        """
        self.tombak.set_time_instruction(tombak.INSTRUCT_PULSE_OUT_WIDTH, 10E-9)
        self.tombak.apply_all()
        
    def close(self):
        """ switches off tombak by setting pulse length to long
        """
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_OUT_WIDTH, 2E64-1)
        self.tombak.apply_all()
        

    def tenmhz(self):
        """ set output frequency 10MHz
        divide by 8 I tried to get div=self.tombak.measure_pulse_in_frequency()/10000000 but
        only works with nearest interger for division
        returns -- 10 MHz aproximately
        """
        div=8
        #div=self.tombak.measure_pulse_in_frequency()/10000000
        #print(div)
        self.divider = div
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_IN_FREQUENCY_DIV,div)
        self.tombak.apply_all()
        return self.get_outputFreqency()
        

# Use the .venv kernal change name please

In [2]:
'''
The max freq output the pulse picker is capable of is 20 MHz mind this when
picking the divider value
'''

'\nThe max freq output the pulse picker is capable of is 20 MHz mind this when\npicking the divider value\n'

In [3]:
device = pp()  # Set 20 ns width, divide by 8


ProtocolError: 

In [ ]:

print("Input freq (Hz):", device.get_inputFreqency())
device.set_outputDivider(80*2)  # Set output divider to 80, so output is 10 MHz
print("Output freq (Hz):", device.get_outputFreqency())


In [ ]:
device = pp()

In [32]:
device 

In [33]:
device.get_inputFreqency()

2002000

In [7]:
device.set_outputDivider(80*2)

NameError: name 'device' is not defined

In [35]:
device.get_outputFreqency()                                                           

12512.5

In [82]:
device.close()

In [36]:
device.tenmhz()

250250.0

In [ ]:
class pp():
    def __init__(self, port='/dev/serial/by-id/usb-FTDI_TTL-232R-3V3-AJ_FTCAVMZN-if00-port0', divider=8, width_ns=20):
        self.tombak = Tombak(port)

        self.tombak.set_status_instruction(self.tombak.INSTRUCT_FUNCTIONING_MODE, self.tombak.DIVIDER)
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_PULSE_IN_SRC, self.tombak.DIRECT)
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_TRIGGER_SRC, self.tombak.INT)
        self.tombak.set_status_instruction(self.tombak.INSTRUCT_SYNC_OUT_1_SRC, self.tombak.PULSE_OUT)

        self.set_output_pulse_width(width_ns)
        self.set_output_divider(divider)

        self.tombak.apply_all()

    def set_output_pulse_width(self, width_ns):
        """Set pulse width in nanoseconds."""
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_OUT_WIDTH, int(width_ns))
        self.tombak.apply_all()

    def set_output_divider(self, div):
        """Set frequency divider."""
        self.divider = div
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_IN_FREQUENCY_DIV, div)
        self.tombak.apply_all()

    def get_input_frequency(self):
        return self.tombak.measure_pulse_in_frequency()

    def get_output_frequency(self):
        return self.get_input_frequency() / self.divider

    def close(self):
        self.tombak.set_integer_instruction(self.tombak.INSTRUCT_PULSE_OUT_WIDTH, int(2**63 - 1))
        self.tombak.apply_all()

    def ten_mhz(self):
        input_freq = self.get_input_frequency()
        div = max(1, round(input_freq / 10e6))
        return self.set_output_divider(div)


In [ ]:
device = pp(divider=8, width_ns=20)  # Set 20 ns width, divide by 8
device.set_output_pulse_width(10)
print("Input freq (Hz):", device.get_input_frequency())
print("Output freq (Hz):", device.get_output_frequency())


